# Load Clean Dataset

In [1]:
import pandas as pd
from pathlib import Path

In [9]:
current_path = Path.cwd().resolve()
root_folder_path = next(
    (
        parent
        for parent in (current_path, *current_path.parents)
        if (parent / "Dataset" / "unzipped" / "anonymisedData" / "courses.csv").is_file()
    ),
    None,
)

if root_folder_path is None:
    raise FileNotFoundError(f"Could not find the project Dataset folder from {current_path}.")

base_path = root_folder_path / "Dataset" / "artefacts"
if not base_path.is_file():
    raise FileNotFoundError(f"Student master CSV was not found at {base_path}.")

In [10]:
df = pd.read_csv(base_path)

In [11]:
list(df.columns)

['code_module',
 'code_presentation',
 'id_student',
 'gender',
 'region',
 'highest_education',
 'imd_band',
 'age_band',
 'num_of_prev_attempts',
 'studied_credits',
 'disability',
 'final_result',
 'total_click',
 'avg_clicks_per_day',
 'max_clicks_day',
 'active_days']

In [3]:
from pathlib import Path

dataset_path = Path(root_folder_path)
print("root_folder_path:", dataset_path)
print("Exists:", dataset_path.exists())
print("Contents:", list(dataset_path.iterdir()) if dataset_path.exists() else "Folder not found")

root_folder_path: D:\
Exists: True
Contents: [WindowsPath('D:/$RECYCLE.BIN'), WindowsPath('D:/100DaysofML_Complete_Course_CampusX.pdf'), WindowsPath('D:/29th March Gen AI CLASS'), WindowsPath('D:/Apna College AI_ML Lecture Notes'), WindowsPath('D:/Apna College Assignments'), WindowsPath('D:/Assignment 1'), WindowsPath('D:/Autogen Crash Course'), WindowsPath('D:/BRAND_GUARDIAN_V1'), WindowsPath('D:/CV'), WindowsPath('D:/Dashboard_Video.mp4'), WindowsPath('D:/DSA'), WindowsPath('D:/DumpStack.log.tmp'), WindowsPath('D:/Flask'), WindowsPath('D:/Full Stack Gen AI'), WindowsPath('D:/Hub_spot'), WindowsPath('D:/LangChain_Complete_Course_OCR.pdf'), WindowsPath('D:/ML practical'), WindowsPath('D:/One Master Prompt You Can Reuse.docx'), WindowsPath('D:/pagefile.sys'), WindowsPath('D:/Personalized_learning_engagement_recommendation_systems'), WindowsPath('D:/PM'), WindowsPath('D:/project_code'), WindowsPath('D:/project_code.zip'), WindowsPath('D:/SNAPCLASS'), WindowsPath('D:/Spring Semester'), Wi

In [4]:
for csv_path in Path(root_folder_path).rglob("*.csv"):
    print(csv_path)

D:\ML practical\thyroid_dataset.csv
D:\Streamlit\sampledata.csv
D:\Apna College AI_ML Lecture Notes\Machine Learning\Linear Regression\HousePricePrediction.csv
D:\Apna College AI_ML Lecture Notes\Machine Learning\Linear Regression\insurance.csv
D:\Assignment 1\ProgrammingAssignment\ProgrammingAssignment_MLAIFNCE9015.csv
D:\BRAND_GUARDIAN_V1\ComplianceQAPipeline\backend\.venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
D:\BRAND_GUARDIAN_V1\ComplianceQAPipeline\backend\.venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-2.csv
D:\BRAND_GUARDIAN_V1\ComplianceQAPipeline\backend\.venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-1.csv
D:\BRAND_GUARDIAN_V1\ComplianceQAPipeline\backend\.venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-2.csv
D:\BRAND_GUARDIAN_V1\ComplianceQAPipeline\backend\.venv\Lib\site-packages\numpy\random\tests\data\pcg64dxsm-testset-1.csv
D:\BRAND_GUARDIAN_V1\ComplianceQAPipeline\backend\.venv\Lib\site-packages\numpy\

# Handle Missing Engagement Values

In [12]:
engagement_cols = ['total_click',
                    'avg_clicks_per_day',
                    'max_clicks_day',
                    'active_days'
                 ]

df[engagement_cols] = df[engagement_cols].fillna(0)

In [13]:
df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,3.337209,23.0,80.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,281.0,3.697368,23.0,12.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,2158.0,3.254902,22.0,123.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,1034.0,2.937500,22.0,70.0


# Understanding the target (final_result)

In [14]:
df["final_result"].value_counts()

final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

# Design Decision

For real-world usefullness, we will do : binary risk predict

Pass, Distinction --> 1 (successfull)
Withdrawn, Fail --> 0 (At risk)

In [15]:
df['success'] = df["final_result"].map({
    "Pass" : 1, 
    "Distinction" : 1,
    "Withdrawn" : 0, 
    "Fail" : 0,
})

In [16]:
df.head(5)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days,success
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0,1
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,3.337209,23.0,80.0,1
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,281.0,3.697368,23.0,12.0,0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,2158.0,3.254902,22.0,123.0,1
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,1034.0,2.937500,22.0,70.0,1


# Engagement Dist Analysis

In [17]:
df.groupby("success")[engagement_cols].mean()

,total_click,avg_clicks_per_day,max_clicks_day,active_days
success,,,,
0,696.286611,2.556024,36.371397,32.541667
1,2354.529477,3.513363,71.822229,98.153526


Successful students --> higher click, more active days

At-risk students --> near zero activity

ML will work in our problem statement.

# Create engagement intensity bands

Instead of raw numers, we will create better/human interpretable categories using feature bucketing.

In [18]:
df.head(1)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days,success
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0,1


In [19]:
df["engagement_level"] = pd.qcut(
    df["total_click"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [20]:
df.head(2)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_click,avg_clicks_per_day,max_clicks_day,active_days,success,engagement_level
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,4.765306,76.0,40.0,1,Medium
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,3.337209,23.0,80.0,1,High


# Absence of engagement

In [21]:
df["no_engagement_flag"] = (df["active_days"] == 0).astype(int)

# 1 --> completely disengaged
# 0 --> some activity

# Save Dataset For ML

In [24]:
output_path = Path(root_folder_path) / "Dataset" / "test" / "artefacts" / "student_master_v2.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)